<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


<h1>Exploring Sentiment Analysis Concepts with a Real-World Dataset</h1>


Estimated time needed: **30** minutes


<h2>Learning Objectives</h2>

In this lab, you will:
<ul>
    <li> Load a sentiment-labeled dataset and inspect class distribution for sequence classification.</li>
    <li> Apply text preprocessing steps to prepare review text for RNN and LSTM sentiment models.</li>
    <li> Identify ambiguous or sarcastic samples and the classification challenges they create, with AI-assisted support.</li>
    <li> Convert review-label pairs into a dataset structure suitable for sequential model training.</li>
    <li> Relate class distribution and data quality issues to expected classifier performance.</li>
</ul> 


## __Table of Contents__

<ul>
    <li><a href="#Import-Libraries">Import Libraries</a></li>
    <li><a href="#Sentiment-Analysis-Workflow">Sentiment Analysis Workflow</a></li>
    <li><a href="#Load-a-Sentiment-labeled-Dataset">Load a Sentiment-labeled Dataset</a></li>
    <li><a href="#Inspect-Class-Distribution">Inspect Class Distribution</a></li>
    <li><a href="#Text-Preprocessing-Workflow">Text Preprocessing Workflow</a></li>
    <li><a href="#Identify-Ambiguous-and-Sarcastic-Samples">Identify Ambiguous and Sarcastic Samples</a></li>
    <li><a href="#Build-a-Vocabulary">Build a Vocabulary</a></li>
    <li><a href="#Prepare-a-Sequential-Dataset">Prepare a Sequential Dataset</a></li>
    <li><a href="#Connect-Data-Quality-to-Model-Performance">Connect Data Quality to Model Performance</a></li>
</ul>


## Import Libraries


First you'll need to install Pytorch and import the required libraries.


In [1]:
# Install PyTorch

!pip install torch

The following required libraries are **not** pre-installed in the Skills Network Labs environment. You will need to run the following cell to import them.


In [2]:
# Import the libraries we need for this lab

import csv
import re
from collections import Counter
from io import StringIO

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(2) 

## Sentiment Analysis Workflow


Sentiment analysis models classify text by emotional polarity, such as positive, negative, or neutral. For RNN and LSTM models, the text must be cleaned, tokenized, converted to numerical sequences, and batched.


<p style="text-align:center">
    <img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/Sketch%20Flow%20Chart%20for%20prediction%20modelling%20-%20sentiment%20analysis.jpg" width="650" alt="Sentiment analysis workflow diagram">
</p>

<p style="text-align:center"><em>Source: Wikimedia Commons, sentiment analysis workflow diagram</em></p>


The quality of the input data strongly affects classifier behavior. Class imbalance, noisy text, sarcasm, and ambiguous labels can reduce model performance.


## Load a Sentiment-labeled Dataset


In [3]:
# Load a sentiment-labeled review dataset

csv_data = """review,label
"The headphones sound crisp and the battery lasts all day",positive
"The delivery was late and the box was damaged",negative
"Customer support solved my issue quickly",positive
"The app freezes every time I try to pay",negative
"The packaging was fine but the product felt average",neutral
"Great, another update that breaks the main feature",negative
"The screen is bright and the setup was simple",positive
"I expected more features for this price",negative
"The color is nice, but the material feels cheap",neutral
"Absolutely perfect if you enjoy waiting forever",negative
"The charger works as described",neutral
"Fast shipping and excellent build quality",positive
"Not bad, but not something I would recommend",neutral
"The keyboard is responsive and comfortable",positive
"The camera quality is terrible in low light",negative
"It does the job without any surprises",neutral
"Wonderful service, because being ignored for two days is wonderful",negative
"The instructions were clear and helpful",positive
"The subscription is expensive for basic features",negative
"The interface is clean and easy to navigate",positive
"The product arrived on time",neutral
"Excellent value and reliable performance",positive
"The fan is loud and distracting",negative
"The design looks modern but feels fragile",neutral
"Setup took five minutes and everything worked",positive
"The refund process was confusing and slow",negative
"The fabric feels soft and durable",positive
"The size was different from the description",negative
"It is okay for occasional use",neutral
"Love how the device randomly shuts down during meetings",negative
"Battery life is decent but charging is slow",neutral
"The display is sharp and colors look accurate",positive
"The buttons stopped working after one week",negative
"The meal was fresh and tasted great",positive
"The room was clean, although the check-in was slow",neutral
"The connection drops whenever I need it most",negative
"The course examples were practical and clear",positive
"The chair looks good but is uncomfortable",negative
"The report exported correctly",neutral
"The update improved speed and stability",positive
"The microphone picks up too much background noise",negative
"The bag has enough space for daily use",positive
"The feature exists, but the workflow is awkward",neutral
"Amazing support if silence counts as support",negative
"The watch tracks workouts accurately",positive
"The order was canceled without explanation",negative
"The hotel location was convenient",positive
"The game is fun but the loading times are long",neutral
"The recipe was easy to follow and delicious",positive
"The printer jams constantly",negative
"The lamp is smaller than expected",neutral
"The staff was professional and polite",positive
"The software crashed before saving my work",negative
"The shoes fit well and feel comfortable",positive
"The website is usable but visually outdated",neutral
"The speaker sounds distorted at high volume",negative
"The onboarding guide answered my questions",positive
"The plan is affordable but missing advanced tools",neutral
"The heater stopped working overnight",negative
"The appointment started on time",positive
"The file upload completed successfully",neutral
"The restaurant forgot half of the order",negative
"The mattress provides excellent support",positive
"The dashboard loads slowly during peak hours",negative
"The sample size is small but useful for practice",neutral
"The backup restored my files correctly",positive
"The headphones are stylish, if style means constant buzzing",negative
"The tutorial was informative",positive
"The remote control response is inconsistent",negative
"The trial version is limited but helpful",neutral
"The search results were relevant and fast",positive
"The package tracking never updated",negative
"The notes synced across devices",positive
"The flavor was mild and acceptable",neutral
"The display cracked after a small drop",negative
"The templates saved a lot of time",positive
"The booking page was hard to use",negative
"The audio quality is acceptable for calls",neutral
"The repair service was quick and honest",positive
"The login page failed repeatedly",negative
"The material feels premium",positive
"The new layout is different but manageable",neutral
"The cable stopped charging after two days",negative
"The analytics view is clear and actionable",positive
"The response time was okay",neutral
"The cleaner left stains on the floor",negative
"The lesson pacing was excellent",positive
"The case protects the phone well",positive
"The lid does not close properly",negative
"The notification settings are basic",neutral
"The experience was smooth from start to finish",positive
"The product works, which is apparently the highest praise now",neutral
"The screen protector bubbles at the edges",negative
"The estimate matched the final invoice",positive
"The menu options are limited",neutral
"The vacuum loses suction quickly",negative
"The class project was engaging and practical",positive
"The account verification email arrived late",negative
"The device is lightweight and portable",positive
"The summary feature is useful but sometimes misses context",neutral
"The headphones disconnect randomly",negative
"The checkout process was simple",positive
"The instructions were missing important steps",negative
"The results page is clear enough",neutral
"The keyboard backlight is bright and adjustable",positive
"The warranty claim was rejected without reason",negative
"The notebook cover is sturdy",positive
"The tool is powerful, though the interface takes practice",neutral
"The screen flickers after sleep mode",negative
"The team responded quickly and fixed the issue",positive
"The form accepted my submission",neutral
"The pizza arrived cold",negative
"The data export was accurate",positive
"The device has potential but feels unfinished",neutral
"The sensor readings are unreliable",negative
"The playlist recommendations were excellent",positive
"The cover fits, but the buttons are harder to press",neutral
"The earbuds hurt after ten minutes",negative
"The documentation examples were clear",positive
"The invoice page loads but formatting is messy",neutral
"The fridge makes a loud clicking sound",negative
"The search filter saved me time",positive
"The product description was misleading",negative
"The response was short but answered the question",neutral
"The backpack straps are comfortable",positive
"The battery drains overnight",negative
"The dashboard summary is helpful",positive
"The color matched the photo",neutral
"The item broke during normal use",negative
"The help article solved the problem",positive
"The installation took longer than expected",negative
"The calendar integration works",neutral
"The trainer explained every step clearly",positive
"The machine overheats under light use",negative
"The interface is acceptable after customization",neutral
"The sound is balanced and rich",positive
"The payment failed without an error message",negative
"The sample report is useful",positive
"The product is fine",neutral
"The strap snapped on the first day",negative
"The staff made the process easy",positive
"The button labels are confusing",negative
"The export option is available",neutral
"The software saves changes automatically",positive
"The lens scratches easily",negative
"The theme looks clean",positive
"The tutorial covers the basics",neutral
"The app logs me out constantly",negative
"The delivery driver was courteous",positive
"The kettle boils water quickly",positive
"The subscription renewal was unclear",negative
"The page has enough information",neutral
"The monitor stand is stable",positive
"The adapter gets too hot",negative
"The settings page is simple",positive
"The product meets minimum expectations",neutral
"The browser extension slowed everything down",negative
"The course quiz reinforced the lesson",positive
"The handle feels loose",negative
"The package included all listed parts",positive
"The feature is useful in some cases",neutral
"The screen brightness changes by itself",negative
"The soup was warm and flavorful",positive
"The update removed a feature I used daily",negative
"The app interface is familiar",neutral
"The customer portal is organized",positive
"The headphones look nice but sound flat",negative
"The installation guide was short",neutral
"The filter works exactly as expected",positive
"The shipment was delayed twice",negative
"The model examples helped me understand the topic",positive
"The account page is functional",neutral
"The charger sparks when plugged in",negative
"The chair supports my back well",positive
"The process is standard",neutral
"The door seal leaks water",negative
"The team handled my request professionally",positive
"The product image was accurate",neutral
"The tablet freezes during video calls",negative
"The guide was concise and useful",positive
"The package arrived",neutral
"The device failed after the first update",negative
"The keyboard feels solid and responsive",positive
"The item was not worth the price",negative
"The report contains the expected fields",neutral
"The support agent was patient and knowledgeable",positive
"The charger cable is too short",negative
"The room temperature was comfortable",positive
"The feedback was mixed",neutral
"The website timed out during payment",negative
"The chart labels are easy to read",positive
"The product does what it says",neutral
"The dishwasher leaves residue on glasses",negative
"The voice assistant understood commands well",positive
"The bag zipper sticks sometimes",neutral
"The update caused repeated errors",negative
"The service appointment was efficient",positive
"The build quality feels average",neutral
"The mobile app drains battery quickly",negative
"The search bar finds exact matches",positive
"The notification arrived",neutral
"The product stopped syncing after setup",negative
"The examples were relevant to my work",positive
"The layout is simple but not elegant",neutral
"The cable frayed after a week",negative
"The checkout discount applied correctly",positive
"The feature works only sometimes",negative
"The default settings are acceptable",neutral
"The device feels durable",positive
"The shipment arrived with missing parts",negative
"The onboarding checklist is helpful",positive
"The message was received",neutral
"The warranty terms are hard to understand",negative
"The playlist loaded instantly",positive
"The case color is darker than expected",neutral
"The motor is noisy",negative
"The recipe instructions were accurate",positive
"The setup wizard completed",neutral
"The item arrived scratched",negative
"The tool made the task easier",positive
"The product is neither impressive nor disappointing",neutral
"The app crashes when opening settings",negative
"The final result exceeded expectations",positive
"The camera is acceptable in daylight",neutral
"The filter broke after one use",negative
"The consultant gave clear recommendations",positive
"The chair assembly was straightforward",positive
"The service fee was higher than advertised",negative
"The dashboard includes basic metrics",neutral
"The screen is responsive and bright",positive
"The software deleted my draft",negative
"The package was plain but secure",neutral
"The headphones have excellent noise cancellation",positive
"The training examples were repetitive",negative
"The product arrived as expected",neutral
"The process was quick and stress free",positive
"The lamp stopped turning on",negative
"The feature is available in the menu",neutral
"The taste was rich and satisfying",positive
"The website form kept resetting",negative
"The result is acceptable",neutral
"The fabric washed well",positive
"The headphones, brilliant, only disconnect every five minutes",negative
"The app has a neutral color theme",neutral
"The replacement arrived quickly",positive
"The product page had inaccurate dimensions",negative
"The export finished",neutral
"The device paired instantly",positive
"The seat cushion flattened after a day",negative
"The interface provides the required controls",neutral
"The search experience is excellent",positive
"The order confirmation never arrived",negative
"The report loaded successfully",neutral
"The training module was clear and engaging",positive
"The handle cracked under normal use",negative
"The workflow is acceptable for small teams",neutral
"The speaker is compact and powerful",positive
"The camera app freezes constantly",negative
"The documentation is complete enough",neutral
"The refund arrived sooner than expected",positive
"The screen has dead pixels",negative
"The service was fine",neutral
"The product made my daily routine easier",positive
"The onboarding emails were confusing",negative
"The file format is supported",neutral
"The response was fast and accurate",positive
"The device loses connection during calls",negative
"The layout works for simple tasks",neutral
"The case survived several drops",positive
"The price increased without notice",negative
"The output matched the sample",neutral
"The lesson examples were easy to adapt",positive
"The app notification sound is too loud",negative
"The product is usable",neutral
"The warranty replacement was hassle free",positive
"The recipe lacked important timing details",negative
"The input form is basic but clear",neutral
"The headphones are comfortable for long sessions",positive
"The package was crushed on arrival",negative
"The generated summary was acceptable",neutral
"The installation team was excellent",positive
"The light flickers randomly",negative
"The chart view is standard",neutral
"The keyboard shortcuts save time",positive
"The tool returned incorrect totals",negative
"The model output needs review",neutral
"The staff went above and beyond",positive
"The charger only works at one angle",negative
"The table is sturdy",positive
"The instructions are average",neutral
"The app deleted my preferences",negative
"The hotel room was quiet and comfortable",positive
"The trial ended earlier than promised",negative
"The package label was readable",neutral
"The headphones provide clear audio",positive
"The support ticket was closed without resolution",negative
"The product page includes basic details",neutral
"The delivery arrived early",positive
"The phone overheats while charging",negative
"The filter options are adequate",neutral
"The model training completed successfully",positive
"The dashboard throws errors after login",negative
"The course overview is concise",neutral
"The device works reliably",positive
"The lid cracked after first use",negative
"The settings saved correctly",neutral
"The interface feels polished",positive
"The purchase confirmation was delayed",negative
"The room had the expected amenities",neutral
"The playlist sync was seamless",positive
"The product failed during setup",negative
"The report is readable",neutral
"The lesson clarified the concept",positive
"The cancellation process was frustrating",negative
"The product is okay",neutral
"The feature reduced manual work",positive
"The headphones stopped pairing",negative
"The response tone was neutral",neutral
"The documentation saved debugging time",positive
"The shipment tracking was inaccurate",negative
"The app opens normally",neutral
"The product feels well made",positive
"The adapter stopped working",negative
"The example covers the main idea",neutral
"The team resolved the issue quickly",positive
"The chair arrived with missing screws",negative
"The result is neither good nor bad",neutral
"The screen is vivid and responsive",positive
"The service was slow and unhelpful",negative
"The package condition was acceptable",neutral
"The guide explained tokenization clearly",positive
"The microphone stopped after one call",negative
"The feature is present but basic",neutral
"The tool improved productivity",positive
"The app froze again, what a delightful surprise",negative
"The delivery status is available",neutral
"The customer care team was excellent",positive
"The product stopped working after warranty ended",negative
"The file opened correctly",neutral
"The final report was accurate and useful",positive
"The login button did nothing",negative
"The user guide is sufficient",neutral
"The headphones sound great",positive
"The battery warning appears too late",negative
"The package included a manual",neutral
"The setup process was smooth",positive
"The filter gave irrelevant results",negative
"The data table is readable",neutral
"The course content was excellent",positive
"The repair took three weeks",negative
"The interface works as expected",neutral
"The headphones stayed connected all day",positive
"The monitor arrived cracked",negative
"The product is standard",neutral
"The examples made the lesson practical",positive
"The checkout page lost my cart",negative
"The text output is acceptable",neutral
"The issue was fixed on the first call",positive
"The phone case feels cheap",negative
"The configuration screen is basic",neutral
"The model predictions were accurate",positive
"The fan stopped spinning",negative
"The document format is supported",neutral
"The packaging protected the item",positive
"The software update failed repeatedly",negative
"The product is fine for the price",neutral
"The support experience was excellent",positive
"The app response is painfully slow",negative
"The settings are easy to find",positive
"The summary is short but usable",neutral
"The feature crashes under normal use",negative
"The keyboard is quiet and responsive",positive
"The invoice details are present",neutral
"The device rebooted during setup",negative
"The delivery experience was professional",positive
"The product documentation is average",neutral
"The charger port is loose",negative
"The analytics dashboard is useful",positive
"The output was generated",neutral
"The restaurant service was rude",negative
"The meal was excellent",positive
"The system message was clear",neutral
"The backup failed without warning",negative
"The chair is comfortable and sturdy",positive
"The text editor lags with large files",negative
"The process completed",neutral
"The headphones have impressive bass",positive
"The replacement part did not fit",negative
"The feature is documented",neutral
"The results were consistent",positive
"The app keeps asking me to log in",negative
"The report title is correct",neutral
"The team communicated clearly",positive
"The product arrived broken",negative
"The page loaded",neutral
"The guide was very helpful",positive
"The search function misses obvious matches",negative
"The interface is okay",neutral
"The delivery was smooth",positive
"The battery stopped charging",negative
"The tool includes basic templates",neutral
"The service exceeded expectations",positive
"The camera lens was scratched",negative
"The form is available online",neutral
"The setup guide made installation easy",positive
"The notification system is unreliable",negative
"The material is acceptable",neutral
"The performance is fast and stable",positive
"The package arrived soaked",negative
"The product description is complete",neutral
"The model checkpoint loaded successfully",positive
"The account page failed to save changes",negative
"The wording is neutral",neutral
"The support answer was detailed",positive
"The headphones crackle during calls",negative
"The result meets the requirement",neutral
"The checkout was quick",positive
"The feature is broken after update",negative
"The screen size is acceptable",neutral
"The design is elegant and practical",positive
"The food was cold and bland",negative
"The item is ordinary",neutral
"The tutorial helped me finish the task",positive
"The printer refuses to connect",negative
"The package was delivered",neutral
"The camera captures sharp images",positive
"The update made the app worse",negative
"The setting is optional",neutral
"The workflow is fast and reliable",positive
"The support chat disconnected",negative
"The item is functional",neutral
"The charger is compact and efficient",positive
"The chair wobbles",negative
"The example is simple",neutral
"The service team was helpful",positive
"The headphones have poor battery life",negative
"The product has a standard design",neutral
"The results were excellent",positive
"The app failed during checkout",negative
"The report includes summary metrics",neutral
"The installation was easy",positive
"The device is defective",negative
"The file was saved",neutral
"The package was neatly packed",positive
"The interface is painfully confusing",negative
"The product is acceptable",neutral
"The training data loaded correctly",positive
"The page is slow",negative
"The system response is neutral",neutral
"The experience was fantastic",positive
"The cable does not fit",negative
"The feature behaves normally",neutral
"The sound quality is excellent",positive
"The update introduced bugs",negative
"The result is fine",neutral
"The customer service was outstanding",positive
"The item smells like chemicals",negative
"The dashboard shows current status",neutral
"The product solved my problem",positive
"The checkout experience was terrible",negative
"The color is standard",neutral
"The course content is well organized",positive
"The screen response is delayed",negative
"The package is intact",neutral
"The tool is intuitive",positive
"The audio cuts out",negative
"The interface has basic navigation",neutral
"The battery lasted all weekend",positive
"The order arrived incomplete",negative
"The label is readable",neutral
"The product feels premium and reliable",positive
"The software freezes constantly",negative
"The summary is adequate",neutral
"The lesson was useful and concise",positive
"The website rejected my valid address",negative
"The output is expected",neutral
"The headphones are excellent",positive
"The account setup failed",negative
"The table format is normal",neutral
"The app is responsive",positive
"The feature does not work",negative
"The product is average",neutral
"The service was prompt",positive
"The package arrived late",negative
"The sample is small",neutral
"The build is solid",positive
"The instructions were unclear",negative
"The result was neutral",neutral
"The workflow saved time",positive
"The headphones stopped working",negative
"The report is complete",neutral
"The experience was excellent",positive
"The item was damaged",negative
"The setting is enabled",neutral
"The delivery was fast",positive
"The login failed",negative
"The product is okay overall",neutral
"The examples are helpful",positive
"The app is unreliable",negative
"The file exists",neutral
"The product works beautifully",positive
"The charger broke",negative
"The interface is standard",neutral
"The support was helpful",positive
"The service was awful",negative
"The item is usable",neutral
"The software is stable",positive
"The package was damaged",negative
"The output is readable",neutral
"The course was excellent",positive
"The app crashed",negative
"The product is normal",neutral
"The results are accurate",positive
"The device failed",negative
"The text is plain",neutral
"The service was great",positive
"The product disappointed me",negative
"The status is available",neutral
"The quality is excellent",positive
"The tool is broken",negative
"The page is simple",neutral
"The experience was good",positive
"The item is poor",negative
"The feature is present",neutral
"The model works well",positive
"The app is bad",negative
"The value is average",neutral
"The product is great",positive
"The service is slow",negative
"The response is neutral",neutral
"The lesson is helpful",positive
"The device is faulty",negative
"The sample is acceptable",neutral
"The interface is clean",positive
"The package is late",negative
"The text is factual",neutral
"The result is strong",positive
"The output is wrong",negative
"The data is valid",neutral
"The support is excellent",positive
"The feature fails",negative
"The status is pending",neutral
"The final experience was positive",positive
"The final experience was negative",negative
"The final experience was neutral",neutral
"""

reader = csv.DictReader(StringIO(csv_data))
dataset_rows = [{"review": row["review"], "label": row["label"]} for row in reader]

print("Number of reviews:", len(dataset_rows))
print(dataset_rows[:3])

Number of reviews: 545
[{'review': 'The headphones sound crisp and the battery lasts all day', 'label': 'positive'}, {'review': 'The delivery was late and the box was damaged', 'label': 'negative'}, {'review': 'Customer support solved my issue quickly', 'label': 'positive'}]


Inspect a few reviews from the dataset:


In [4]:
# Preview reviews and labels

for row in dataset_rows[:5]:
    print(row["label"], "-", row["review"])

positive - The headphones sound crisp and the battery lasts all day
negative - The delivery was late and the box was damaged
positive - Customer support solved my issue quickly
negative - The app freezes every time I try to pay
neutral - The packaging was fine but the product felt average


## Inspect Class Distribution


Class distribution shows how many examples belong to each sentiment category. Imbalanced data can bias a model toward the majority class.


In [5]:
# Count examples by class

label_counts = Counter(row["label"] for row in dataset_rows)

print(label_counts)

Counter({'positive': 190, 'negative': 188, 'neutral': 167})


Convert counts into percentages:


In [6]:
# Compute class distribution percentages

total_examples = len(dataset_rows)

for label, count in label_counts.items():
    percentage = count / total_examples * 100
    print(label, round(percentage, 2), "%")

positive 34.86 %
negative 34.5 %
neutral 30.64 %


## Text Preprocessing Workflow


Preprocessing prepares raw review text for sequential models. Common steps include lowercasing, removing punctuation, tokenizing, and measuring sequence length.


In [7]:
# Create a preprocessing function

def preprocess_review(text):
    text = text.lower()
    tokens = re.findall(r"\b\w+\b", text)
    return tokens

Apply preprocessing to each review:


In [8]:
# Preprocess all reviews

for row in dataset_rows:
    row["tokens"] = preprocess_review(row["review"])
    row["length"] = len(row["tokens"])

print(dataset_rows[0])

{'review': 'The headphones sound crisp and the battery lasts all day', 'label': 'positive', 'tokens': ['the', 'headphones', 'sound', 'crisp', 'and', 'the', 'battery', 'lasts', 'all', 'day'], 'length': 10}


Inspect sequence lengths to plan padding and truncation:


In [9]:
# Analyze review lengths

lengths = [row["length"] for row in dataset_rows]

print("Shortest review:", min(lengths))
print("Longest review:", max(lengths))
print("Average length:", round(sum(lengths) / len(lengths), 2))

Shortest review: 3
Longest review: 10
Average length: 5.42


## Identify Ambiguous and Sarcastic Samples


Ambiguous and sarcastic reviews can be difficult for sentiment classifiers because literal words may conflict with the intended meaning. AI-assisted support can help flag samples for review, but human validation is still needed.


In [10]:
# Create simple AI-assisted review flags

positive_words = {"great", "perfect", "wonderful", "amazing", "love", "excellent", "good", "helpful"}
contrast_words = {"but", "although", "though", "however"}

def flag_review_challenges(row):
    tokens = set(row["tokens"])
    flags = []
    if row["label"] == "negative" and tokens.intersection(positive_words):
        flags.append("possible sarcasm")
    if tokens.intersection(contrast_words):
        flags.append("mixed sentiment or contrast")
    if row["length"] <= 3:
        flags.append("short context")
    return flags

Apply the flags and inspect challenging examples:


In [11]:
# Find ambiguous or sarcastic samples

challenging_rows = []

for row in dataset_rows:
    row["flags"] = flag_review_challenges(row)
    if row["flags"]:
        challenging_rows.append(row)

for row in challenging_rows[:10]:
    print(row["label"], row["flags"], "-", row["review"])

neutral ['mixed sentiment or contrast'] - The packaging was fine but the product felt average
negative ['possible sarcasm'] - Great, another update that breaks the main feature
neutral ['mixed sentiment or contrast'] - The color is nice, but the material feels cheap
negative ['possible sarcasm'] - Absolutely perfect if you enjoy waiting forever
neutral ['mixed sentiment or contrast'] - Not bad, but not something I would recommend
negative ['possible sarcasm'] - Wonderful service, because being ignored for two days is wonderful
neutral ['mixed sentiment or contrast'] - The design looks modern but feels fragile
negative ['possible sarcasm'] - Love how the device randomly shuts down during meetings
neutral ['mixed sentiment or contrast'] - Battery life is decent but charging is slow
neutral ['mixed sentiment or contrast'] - The room was clean, although the check-in was slow


These flags are not final labels. They are prompts for review because sarcasm and mixed sentiment can confuse classifiers trained only on surface word patterns.


## Build a Vocabulary


Build a vocabulary so each token can be mapped to an integer ID. Use special tokens for padding and unknown words.


In [12]:
# Build token counts

token_counts = Counter(token for row in dataset_rows for token in row["tokens"])

print("Unique tokens:", len(token_counts))
print(token_counts.most_common(10))

Unique tokens: 846
[('the', 578), ('is', 156), ('was', 101), ('and', 52), ('product', 34), ('but', 24), ('app', 19), ('excellent', 19), ('are', 19), ('arrived', 18)]


Create token-to-index and label-to-index mappings:


In [13]:
# Create vocabulary and label mappings

special_tokens = ["<pad>", "<unk>"]
vocab = {token: index for index, token in enumerate(special_tokens)}

for token in token_counts:
    if token not in vocab:
        vocab[token] = len(vocab)

label_to_index = {"negative": 0, "neutral": 1, "positive": 2}
pad_id = vocab["<pad>"]
unk_id = vocab["<unk>"]

print("Vocabulary size:", len(vocab))
print("Label mapping:", label_to_index)

Vocabulary size: 848
Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}


## Prepare a Sequential Dataset


Sequential models such as RNNs and LSTMs expect review text as ordered token ID sequences with labels.


In [14]:
# Encode tokens as token IDs

def encode_tokens(tokens, vocabulary):
    return [vocabulary.get(token, unk_id) for token in tokens]

Create a custom PyTorch Dataset:


In [15]:
# Create a Dataset for sequential sentiment modeling

class SentimentReviewDataset(Dataset):
    def __init__(self, rows, vocabulary, label_mapping):
        self.rows = rows
        self.vocabulary = vocabulary
        self.label_mapping = label_mapping

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        token_ids = encode_tokens(row["tokens"], self.vocabulary)
        label_id = self.label_mapping[row["label"]]
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label_id, dtype=torch.long)


sentiment_dataset = SentimentReviewDataset(dataset_rows, vocab, label_to_index)

print("Dataset size:", len(sentiment_dataset))
print(sentiment_dataset[0])

Dataset size: 545
(tensor([ 2,  3,  4,  5,  6,  2,  7,  8,  9, 10]), tensor(2))


Use a custom collate function to pad variable-length reviews inside each batch:


In [16]:
# Create a collate function for padded batches

def collate_reviews(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=pad_id)
    labels = torch.stack(labels)
    return padded_sequences, labels, lengths

Create a DataLoader and inspect one batch:


In [17]:
# Create a DataLoader

loader = DataLoader(sentiment_dataset, batch_size=8, shuffle=True, collate_fn=collate_reviews)

batch_sequences, batch_labels, batch_lengths = next(iter(loader))

print("Sequences shape:", batch_sequences.shape)
print("Labels shape:", batch_labels.shape)
print("Lengths:", batch_lengths)

Sequences shape: torch.Size([8, 9])
Labels shape: torch.Size([8])
Lengths: tensor([5, 4, 4, 4, 9, 5, 4, 5])


## Connect Data Quality to Model Performance


Class distribution and data quality issues shape expected classifier performance. A model may learn majority-class patterns, struggle with short context, or misclassify sarcastic reviews that contain positive words but negative intent.


In [18]:
# Summarize data quality signals

quality_summary = {
    "total_reviews": len(dataset_rows),
    "class_counts": dict(label_counts),
    "challenging_reviews": len(challenging_rows),
    "min_length": min(lengths),
    "max_length": max(lengths),
    "average_length": round(sum(lengths) / len(lengths), 2),
}

quality_summary

{'total_reviews': 545,
 'class_counts': {'positive': 190, 'negative': 188, 'neutral': 167},
 'challenging_reviews': 43,
 'min_length': 3,
 'max_length': 10,
 'average_length': 5.42}

Use this summary before training. If one class dominates or many samples are ambiguous, evaluation metrics should be interpreted carefully.



<a href="https://dataplatform.cloud.ibm.com/registration/stepone?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork&context=cpdaas&apps=data_science_experience%2Cwatson_machine_learning"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork/Template/module%201/images/Watson_Studio.png"></a>


## Author(s)


**Harish Pant** is an accomplished IT Consultant specializing in bridging the gap between complex AI technologies and practical business applications. Holding a Bachelor’s degree in Computer Science with a strong background in Machine Learning, Harish serves clients across diverse industries—including fintech, retail, and manufacturing—helping them navigate AI adoption. Harish is known for translating technical AI concepts into actionable business strategies and delivering engaging, high-impact presentations.



<!--Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2026-05-29|0.1|Patsy Kravitz|Initial publishing of the lab by Harish Pant|-->


## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
